# Week 2 Sensitivity Checks

This notebook supports two checks from Project Plan v9:

1. MAD-proxy vs textbook MAD Spearman rank correlation.
2. `min_liquidity` sweep over `{0.25, 0.5, 1.0, 2.0, 5.0}`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from market_anomaly_engine import MarketAnomalyEngine

BASE = Path.cwd().parents[1]
DATA_PATH = BASE / 'data' / 'processed' / 'week2_sample_prices.csv'
DATA_PATH

In [ ]:
def textbook_mad_scores(frame: pd.DataFrame, window: int = 20) -> pd.DataFrame:
    out = frame.copy()
    log_vol = np.log1p(out['Volume'])
    out['Volume_Textbook_MAD'] = log_vol.rolling(window).apply(
        lambda values: np.median(np.abs(values - np.median(values))),
        raw=True,
    )
    out['Volume_Textbook_Z'] = (log_vol - log_vol.rolling(window).median()) / (1.4826 * out['Volume_Textbook_MAD'] + 1e-8)

    out['Returns'] = np.log(out['Adj_Close'] / out['Adj_Close'].shift(1))
    out['Returns_Textbook_MAD'] = out['Returns'].rolling(window).apply(
        lambda values: np.median(np.abs(values - np.median(values))),
        raw=True,
    )
    out['Returns_Textbook_Z'] = (out['Returns'] - out['Returns'].rolling(window).median()) / (1.4826 * out['Returns_Textbook_MAD'] + 1e-8)
    return out

def spearman_proxy_vs_textbook(frame: pd.DataFrame, window: int = 20) -> dict[str, float]:
    engine = MarketAnomalyEngine(window=window)
    proxy = engine.analyze_ticker_data(frame)
    textbook = textbook_mad_scores(frame, window=window)

    vol_mask = proxy['Vol_ZScore_Robust'].notna() & textbook['Volume_Textbook_Z'].notna()
    ret_mask = proxy['Volat_ZScore_Robust'].notna() & textbook['Returns_Textbook_Z'].notna()

    vol_corr = spearmanr(proxy.loc[vol_mask, 'Vol_ZScore_Robust'].abs(), textbook.loc[vol_mask, 'Volume_Textbook_Z'].abs()).correlation
    ret_corr = spearmanr(proxy.loc[ret_mask, 'Volat_ZScore_Robust'].abs(), textbook.loc[ret_mask, 'Returns_Textbook_Z'].abs()).correlation
    return {'volume_spearman': float(vol_corr), 'returns_spearman': float(ret_corr)}

In [ ]:
def min_liquidity_sweep(frame: pd.DataFrame, thresholds=(0.25, 0.5, 1.0, 2.0, 5.0), window: int = 20) -> pd.DataFrame:
    rows = []
    for threshold in thresholds:
        engine = MarketAnomalyEngine(window=window, min_liquidity=threshold)
        scored = engine.analyze_ticker_data(frame)
        rows.append({
            'min_liquidity': threshold,
            'volume_illiquid_rows': int(scored['Low_Liquidity_Volume_Flag'].sum()),
            'returns_illiquid_rows': int(scored['Low_Liquidity_Returns_Flag'].sum()),
            'volume_anomalies': int(scored['Is_Volume_Anomaly'].sum()),
            'volatility_anomalies': int(scored['Is_Volatility_Anomaly'].sum()),
        })
    return pd.DataFrame(rows)

if DATA_PATH.exists():
    sample = pd.read_csv(DATA_PATH)
    display(spearman_proxy_vs_textbook(sample))
    display(min_liquidity_sweep(sample))
else:
    print(f'Place a sample price file at {DATA_PATH} to run the checks.')

## Notes

- Use adjusted close, not raw close.
- Run this on 5-10 representative tickers for the Week 2 report.
- Record the Spearman correlation and threshold sweep table in the write-up.